# Experiment Hard Sample Mining

This notebook focuses on implementing _hard sample mining_ -- a training method that increases the probability of a sample being shown to the network if the sample had a high loss in the previous iterations.

In [ ]:
%reload_ext autoreload
%autoreload 2
%load_ext line_profiler

import os
from dataclasses import dataclass
from pathlib import Path
from pprint import pprint

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from albumentations.pytorch import ToTensorV2
from clearml import Task
from dotenv import load_dotenv
from torch import nn
from torch.utils.data import DataLoader

from mglyph_ml.dataset.glyph_dataset import GlyphDataset
from mglyph_ml.dataset.utils import load_split_samples, show_datasets
from mglyph_ml.experiment.run_config import RunConfigBase
from mglyph_ml.experiment.visualization import show_loss_vs_x_graph, show_truth_vs_pred_graph
from mglyph_ml.nn.glyph_regressor_binned import BinnedGlyphRegressor

# Hyperparameters

Every experiment can run using some set of parameters. These parameters can vary literally anything in the experiment. They can change the shape of the neural network, they can vary the training set, set the learning rate, choose an optimizer, or simply set a random number generator seed. The sky is the limit here.

Notice that this cell has the `parameters` tag associated with it. That tag is very important, because it acts as a 'hook' for a tool called Papermill. This tool injects custom parameters into the notebook and runs it. Injecting parameters is useful because it allows for the execution of series of experiments. We can run the notebook 20 times with different parameters, and analyze the results.

When the notebook is run, it exports important information to the [ClearML web server](https://app.clear.ml/), where the data is aggregated, and can be analyzed. Some parameters like `task_tag` and `task_name` refer to the ClearML Task properties.

*`scheduler_gamma`:*

The gamma is calculated using this formula: $\text{total\_multiplier}^{\frac{1}{\text{total\_steps}}}$. The `total_decrease` corresponds to the total multiplier of the initial learning rate that we wish to be applied, and `total_steps` represents the number of steps necessary to achieve this multiplier. So, for example, if we want our learning rate to decay by a factor of 10 in 1000 steps, we can use $\text{gamma} = \frac{1}{10}^\frac{1}{1000} = \sqrt[1000]{\frac{1}{10}}$.

In [ ]:
@dataclass(frozen=True)
class RunConfig(RunConfigBase):
    # The ClearML task tag (so we can easily filter this experiment in the ClearML web GUI).
    task_tag: str = "stability-2"

    # Where the dataset lies.
    dataset_path: Path = Path("data/simple-star-10k.mglyph")

    # This seed is used by (almost) every single RNG in the experiment. PyTorch's backend isn't set to
    # deterministic mode because it makes it slightly slower. But dataset sampling is determined by the seed
    # and many other stochastic operations
    seed: int = 325

    # We don't have epochs, we only use "steps". 1 step corresponds to the training of the NN on a single batch.
    max_steps: int = 1000

    # The start learning rate. It changes during the experiment because of the LR scheduler.
    learning_rate: float = 0.0005

    # Whether to report the task to ClearML or not. Set to True to stop reporting to ClearML.
    offline: bool = True

    # Size of a single batch that is processed in one step.
    batch_size: int = 64

    # In binned regression, this corresponds to the number of divisions of the [0.0; 100.0] interval.
    num_divisions: int = 5

    # This parameter is used to reduce the number of samples used during training, for debugging purposes.
    sample_count: int = 9999999999

    # Hard sample mining parameters.
    hard_mining_enabled: bool = True
    hard_mining_warmup_steps: int = 200
    hard_mining_candidate_multiplier: int = 2
    hard_mining_random_fraction: float = 0.5
    hard_mining_score_ema_alpha: float = 0.10

    # Edge-aware training parameters.
    edge_focus_enabled: bool = True
    edge_band_inner: float = 2.0
    edge_band_outer: float = 8.0
    edge_quota_fraction: float = 0.25
    edge_near_fraction: float = 0.25
    edge_weight_alpha: float = 3.0
    edge_weight_tau: float = 3.0
    edge_weight_max: float = 8.0

    # How fast the learning rate decays.
    # scheduler_gamma: float = (1 / 10) ** (1 / 1000)

    # Optional explicit name; if None, it is derived from other fields below.
    task_name: str = f"Hard sample mining [seed={seed}]"

# Here, we clear the global variables that have the same names as the fields in this RunConfig class.
RunConfig.clear_globals()

In [ ]:
# This is the global RunConfig instance. You can use c.xxx to access the `xxx` parameter.
c = RunConfig.from_globals()

Task.set_offline(c.offline)
task: Task = Task.init(project_name="mglyph-ml", task_name=c.task_name)
task.add_tags(c.task_tag)
task.connect(c)

load_dotenv()
pprint(c)

# Data Loading

The data is loaded from the dataset file from so-called _splits_. You can think of a split like a folder inside the dataset; it's simply used to group a bunch of samples together. A single sample inside the dataset has these properties:

- `label`: the real x-value of the glyph (sample)
- `image`: this is the binary data of the glyph itself
- (optional) `metadata`: this is an arbitrary `dict` containing some useful information the creator of the dataset wanted to embed into it

In [ ]:
loaded_split_0 = load_split_samples(
    dataset_path=c.dataset_path,
    split="0",
    desired_size=(64, 64),
    shuffle=True,
    seed=c.seed,
)

train_limit = min(c.sample_count, len(loaded_split_0))
samples_train = loaded_split_0[:train_limit]
images_train = [s.image for s in samples_train]
labels_train = [s.label for s in samples_train]

# Augmentation

When training the neural network, we can _augment_ the samples, which is a fancy word for modifying them a bit to make the samples a bit harder for the neural network to learn from. In this simple case, we simply rotate them a bit and move them around a tiny bit. This way, the NN cannot rely on simple patterns like a single line through the image, but has to find other, more complex ways to predict the correct label.

# GlyphDataset

This class is a data structure that holds a series of samples and labels, and applies some kind of transformation to them. We need this class because PyTorch provides a `DataLoader` class whose constructor needs an instance of a `Dataset` class. By following PyTorch's recommended data loading procedures, we gain access to automatic optimizations which include caching, thus, making the training faster.

In [ ]:
affine = A.Affine(
    rotate=(-1, 1),
    translate_percent={"x": (-0.01, 0.01), "y": (-0.01, 0.01)},
    fit_output=False,
    keep_ratio=True,
    border_mode=cv2.BORDER_CONSTANT,
    fill=255,
    p=1.0,
)
normalize = A.Normalize(mean=0.0, std=1.0, max_pixel_value=255.0)
to_tensor = ToTensorV2()
pipeline = A.Compose([affine, normalize, to_tensor], seed=c.seed)
normalize_pipeline = A.Compose([normalize, to_tensor])


def affine_and_normalize(image: np.ndarray) -> torch.Tensor:
    return pipeline(image=image)["image"]


def just_normalize(image: np.ndarray) -> torch.Tensor:
    return normalize_pipeline(image=image)["image"]


dataset_train = GlyphDataset(
    name="Training",
    images=images_train,
    labels=labels_train,
    transform=affine_and_normalize,
)

print(f"train dataset size: {len(dataset_train)}")

In [ ]:
show_datasets(dataset_train)

# Model

Here, we initialize the model and other important objects required during training. The DataLoader is also created here. The reason why it is not created right after the creation of the `GlyphDataset` is because we want to re-create the DataLoader when we want to re-create the model from scratch. The `GlyphDataset` never changes, it holds no mutable state. However, the `DataLoader` needs a seed which feeds a RNG inside the `DataLoader`. If we want to deterministically re-create the training, we need to reset the `DataLoader`.

The `scheduler` is an object responsible for the changing of the learning rate during training.

In [ ]:
device = os.environ["MGML_DEVICE"]
data_loader_num_workers = int(os.environ["DATA_LOADER_NUM_WORKERS"])
print(f"Training device: {device}")

model = BinnedGlyphRegressor(num_divisions=c.num_divisions)
criterion_train = nn.MSELoss(reduction="none")
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=c.learning_rate)
# scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=c.scheduler_gamma)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=700, gamma=0.5)
generator = torch.Generator().manual_seed(c.seed)

data_loader_train = DataLoader(
    dataset_train,
    batch_size=c.batch_size,
    num_workers=data_loader_num_workers,
    pin_memory=True,
    persistent_workers=True,
    shuffle=True,
    generator=generator,
)

# Training

The training is done in _steps_. One step corresponds to a single batch processed by the NN (with the update of the NN's weights). In this example, we don't even have epochs. Everything is done in steps. At the beginning of the cell, we create a `train_iterator`, which is an iterator that can fetch training samples from the training `DataLoader`. Once this iterator is exhausted, it's reset to the beginning and looped through again. This way, we can train indefinitely and the learning curve is not dependent on the amount of samples we're training on. However, it's still dependent on the batch size. If we increase the batch side, the number of steps required to go through the entire training set gets lower, because we're crunching through the dataset at a quicker rate.

In [ ]:
from collections import deque

model.to(device)

step = 0
running_loss_train = 0.0
running_error_train = 0.0
num_batches_train = 0

recent_train_losses = deque(maxlen=20)
recent_full_losses = deque(maxlen=20)

dataset_size = len(dataset_train)
if dataset_size < c.batch_size:
    raise ValueError(f"Dataset too small for batch_size={c.batch_size}. Size={dataset_size}")

rng = np.random.default_rng(c.seed)
all_indices = np.arange(dataset_size)
labels_arr = np.asarray(labels_train, dtype=np.float32)
sample_scores = np.zeros(dataset_size, dtype=np.float32)

# Diagnostics buffers (how often each sample is picked overall/random/hard).
selected_total_count = np.zeros(dataset_size, dtype=np.int32)
selected_random_count = np.zeros(dataset_size, dtype=np.int32)
selected_hard_count = np.zeros(dataset_size, dtype=np.int32)

edge_inner = float(c.edge_band_inner)
edge_outer = float(c.edge_band_outer)
if not (0.0 <= edge_inner <= edge_outer <= 50.0):
    raise ValueError("Edge bands must satisfy 0 <= inner <= outer <= 50")

edge_mask = (labels_arr <= edge_inner) | (labels_arr >= 100.0 - edge_inner)
near_mask = ((labels_arr > edge_inner) & (labels_arr <= edge_outer)) | (
    (labels_arr < 100.0 - edge_inner) & (labels_arr >= 100.0 - edge_outer)
 )
mid_mask = ~(edge_mask | near_mask)

edge_indices = all_indices[edge_mask]
near_indices = all_indices[near_mask]
mid_indices = all_indices[mid_mask]


def sample_without_replacement(pool: np.ndarray, count: int, exclude: np.ndarray | None = None) -> np.ndarray:
    if count <= 0 or len(pool) == 0:
        return np.array([], dtype=np.int64)

    if exclude is not None and len(exclude) > 0:
        pool = np.setdiff1d(pool, exclude, assume_unique=False)

    if len(pool) == 0:
        return np.array([], dtype=np.int64)

    take = min(count, len(pool))
    return rng.choice(pool, size=take, replace=False)


def build_batch(indices: np.ndarray) -> tuple[torch.Tensor, torch.Tensor]:
    imgs = []
    labs = []
    for sample_idx in indices.tolist():
        image_tensor, label_tensor = dataset_train[sample_idx]
        imgs.append(image_tensor)
        labs.append(float(label_tensor))

    inputs_tensor = torch.stack(imgs, dim=0).to(device, non_blocking=True).float()
    labels_tensor = torch.tensor(labs, dtype=torch.float32, device=device)
    return inputs_tensor, labels_tensor


def sample_mixed_batch_indices() -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    edge_count = int(c.batch_size * c.edge_quota_fraction) if c.edge_focus_enabled else 0
    near_count = int(c.batch_size * c.edge_near_fraction) if c.edge_focus_enabled else 0
    edge_count = max(0, min(edge_count, c.batch_size))
    near_count = max(0, min(near_count, c.batch_size - edge_count))

    chosen_edge = sample_without_replacement(edge_indices, edge_count)
    chosen_near = sample_without_replacement(near_indices, near_count, exclude=chosen_edge)
    guaranteed = np.concatenate([chosen_edge, chosen_near])

    remaining = c.batch_size - len(guaranteed)
    if remaining <= 0:
        random_selected = guaranteed
        hard_selected = np.array([], dtype=np.int64)
        return guaranteed, random_selected, hard_selected

    available_pool = np.setdiff1d(all_indices, guaranteed, assume_unique=False)
    random_count = int(remaining * c.hard_mining_random_fraction)
    hard_count = remaining - random_count

    random_part = sample_without_replacement(available_pool, random_count)

    if (
        not c.hard_mining_enabled
        or step < c.hard_mining_warmup_steps
        or hard_count <= 0
    ):
        fill_pool = np.setdiff1d(available_pool, random_part, assume_unique=False)
        fill = sample_without_replacement(fill_pool, hard_count)
        batch = np.concatenate([guaranteed, random_part, fill])
        random_selected = np.concatenate([guaranteed, random_part, fill])
        hard_selected = np.array([], dtype=np.int64)
        return batch, random_selected, hard_selected

    candidate_count = min(
        len(available_pool) - len(random_part),
        c.hard_mining_candidate_multiplier * remaining,
    )
    if candidate_count < hard_count:
        fill_pool = np.setdiff1d(available_pool, random_part, assume_unique=False)
        fill = sample_without_replacement(fill_pool, hard_count)
        batch = np.concatenate([guaranteed, random_part, fill])
        random_selected = np.concatenate([guaranteed, random_part, fill])
        hard_selected = np.array([], dtype=np.int64)
        return batch, random_selected, hard_selected

    hard_pool = np.setdiff1d(available_pool, random_part, assume_unique=False)
    candidates = sample_without_replacement(hard_pool, candidate_count)
    candidate_scores = sample_scores[candidates]
    hardest_local_indices = np.argpartition(candidate_scores, -hard_count)[-hard_count:]
    hard_part = candidates[hardest_local_indices]

    batch = np.concatenate([guaranteed, random_part, hard_part])
    random_selected = np.concatenate([guaranteed, random_part])
    hard_selected = hard_part
    return batch, random_selected, hard_selected


while step < c.max_steps:
    model.train()

    batch_indices, random_part, hard_part = sample_mixed_batch_indices()

    selected_total_count[batch_indices] += 1
    selected_random_count[random_part] += 1
    if len(hard_part) > 0:
        selected_hard_count[hard_part] += 1

    inputs, labels = build_batch(batch_indices)

    optimizer.zero_grad()
    logits: torch.Tensor = model(inputs)
    preds = model.logits_to_labels(logits)

    per_sample_loss = criterion_train(preds, labels)
    full_batch_loss = per_sample_loss.mean()

    if c.edge_focus_enabled:
        weights = 1.0 + c.edge_weight_alpha * torch.exp(-labels / c.edge_weight_tau)
        weights = weights + c.edge_weight_alpha * torch.exp(-(100.0 - labels) / c.edge_weight_tau)
        if c.edge_weight_max > 0:
            weights = torch.clamp(weights, max=c.edge_weight_max)
        loss = torch.mean(weights * per_sample_loss)
        hardness_signal = (weights * per_sample_loss).detach().cpu().numpy()
    else:
        loss = full_batch_loss
        hardness_signal = per_sample_loss.detach().cpu().numpy()

    loss.backward()
    optimizer.step()
    scheduler.step()

    sample_scores[batch_indices] = (
        (1.0 - c.hard_mining_score_ema_alpha) * sample_scores[batch_indices]
        + c.hard_mining_score_ema_alpha * hardness_signal
    )

    error = torch.mean(torch.abs(preds - labels)).item()

    running_loss_train += loss.item()
    running_error_train += error
    num_batches_train += 1
    recent_train_losses.append(loss.item())
    recent_full_losses.append(full_batch_loss.item())
    step += 1

    if step % 100 == 0 or step == c.max_steps:
        avg_recent_loss = sum(recent_train_losses) / len(recent_train_losses)
        avg_recent_full_loss = sum(recent_full_losses) / len(recent_full_losses)

        if not c.hard_mining_enabled:
            phase = "random-only"
        elif step <= c.hard_mining_warmup_steps:
            phase = "warmup-random"
        else:
            phase = "hard-mining"

        print("=" * 80)
        print(
            f"Step {step} [{phase}]: objective(last {len(recent_train_losses)} avg)={avg_recent_loss:.6f}, "
            f"full_batch_mse(last {len(recent_full_losses)} avg)={avg_recent_full_loss:.6f}, "
            f"lr={scheduler.get_last_lr()[0]:.6f}"
        )

In [ ]:
# Diagnostics: verify how hard-sample mining changed sampling behavior.
required_vars = [
    "selected_total_count",
    "selected_random_count",
    "selected_hard_count",
    "sample_scores",
    "labels_train",
]

missing = [name for name in required_vars if name not in globals()]
if missing:
    print("Missing diagnostics state:", missing)
    print("Re-run Cell 14 (Training) first, then run this diagnostics cell again.")
else:
    total_draws = int(selected_total_count.sum())
    random_draws = int(selected_random_count.sum())
    hard_draws = int(selected_hard_count.sum())
    selected_any = int(np.count_nonzero(selected_total_count))

    print(f"Total sampled positions: {total_draws}")
    print(f"Random sampled positions: {random_draws}")
    print(f"Hard sampled positions:   {hard_draws}")
    if total_draws > 0:
        print(f"Hard share: {hard_draws / total_draws:.4f}")
    print(f"Unique samples selected at least once: {selected_any}/{len(selected_total_count)}")

    top_k = min(12, len(selected_total_count))
    top_total_idx = np.argpartition(selected_total_count, -top_k)[-top_k:]
    top_total_idx = top_total_idx[np.argsort(selected_total_count[top_total_idx])[::-1]]

    print("\nTop selected samples (index, x, total, random, hard, score):")
    for idx in top_total_idx.tolist():
        print(
            f"  i={idx:5d}, x={labels_train[idx]:8.4f}, total={selected_total_count[idx]:5d}, "
            f"random={selected_random_count[idx]:5d}, hard={selected_hard_count[idx]:5d}, "
            f"score={sample_scores[idx]:.6f}"
        )

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(selected_total_count, bins=30, alpha=0.8, label="total")
    axes[0].set_title("Selection count distribution")
    axes[0].set_xlabel("times selected")
    axes[0].set_ylabel("num samples")

    labels_arr = np.asarray(labels_train, dtype=np.float64)
    axes[1].scatter(labels_arr, selected_hard_count, s=8, alpha=0.4)
    axes[1].set_title("Hard picks vs label x")
    axes[1].set_xlabel("x label")
    axes[1].set_ylabel("hard-selected count")

    axes[2].scatter(selected_hard_count, sample_scores, s=8, alpha=0.4)
    axes[2].set_title("Hard picks vs current score")
    axes[2].set_xlabel("hard-selected count")
    axes[2].set_ylabel("sample score (EMA loss)")

    fig.tight_layout()
    plt.show()

Here, we show a graph which shows the real value of x vs. the predicted one. All the points on this graph should fall as close to the $y = x$ line as possible.

In [ ]:
fig, ax = show_truth_vs_pred_graph(
    n_samples=2500,
    model=model,
    seed=c.seed,
    dataset=dataset_train,
    device=device,
)

In [ ]:
fig, ax, worst_5_loss_avg, worst_5_losses = show_loss_vs_x_graph(
    n_samples=2500,
    model=model,
    dataset=dataset_train,
    device=device,
    loss_fn=criterion,
    seed=c.seed,
)

task.logger.report_scalar(
    title="Evaluation",
    series="Worst 5 Loss Avg",
    value=worst_5_loss_avg,
    iteration=c.max_steps,
)

print(f"Worst 5 losses: {worst_5_losses}")
print(f"Worst 5 average loss: {worst_5_loss_avg:.6f}")

In [ ]:
task.flush(wait_for_uploads=True)
task.close()